# Can Search Signals Prioritize Content Refresh Reviews?

## An honest client-grouped evaluation of a refresh opportunity queue

**Author:** Batuhan Şahin  
**Lane:** Refresh / Content Opportunity Scoring  
**Repository:** [Cooper30/flyrank-ml-internship](https://github.com/Cooper30/flyrank-ml-internship)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cooper30/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

## Abstract

This study asks whether March 2026 search and content signals can prioritize pages for editorial refresh review before a substantial April impressions decline. Using the FlyRank ML Internship warehouse, I built a 101,280-row modeling frame across 40 pseudonymized clients and a separate March action queue covering 176,738 pages. A shallow Decision Tree using five pre-outcome features was compared with a transparent age-and-visibility rule under a client-grouped 60/20/20 train, validation, and test design. On 46,850 held-out test rows, the model improved Precision@50 from 0.26 to 0.38 and ROC-AUC from 0.467 to 0.570, but its Precision@50 remained below the 0.558 test base rate and validation behavior was unstable. Because the learned ranking was not reliable enough for automation, the final playbook uses a transparent rule with reason codes and mandatory human review to rank pages as review now, review next, monitor, or defer.

## 1. Introduction / Problem statement

An editor cannot inspect every page at once. The practical decision is which existing pages deserve attention first when editorial time is limited. The unit of analysis is one pseudonymized client-content item, and the intended output is an ordered review queue with a reason code that a person can challenge.

A wrong high-priority call wastes editor time; a wrong low-priority call can leave a valuable page unattended. The system therefore supports prioritization rather than replacing editorial judgment.

## 2. Data

The analysis uses the gated FlyRank Internship Warehouse release (build `20260703`), primarily `fact_content_daily_performance` and `dim_content`. March 1–31, 2026 is the observation window, and April 1–30, 2026 is the model outcome window. The action queue uses March-only information and contains 176,738 pages; the model frame applies additional coverage and minimum-volume requirements and contains 101,280 rows across 40 pseudonymized clients.

The label is one when April impressions are below 80% of March impressions. Modeling requires at least 100 March impressions and at least 20 available GSC days for the client in both March and April. The five model features are log March impressions, March CTR, March average position, March observed days, and content age.

Client/content IDs are used only for grouping and joining. April fields, label-derived values, product flags, client names, domains, URLs, raw queries, credentials, and private exports are excluded from model features and public outputs.

In [ ]:
import json
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BASE = "https://raw.githubusercontent.com/Cooper30/flyrank-ml-internship/main/work/outputs"
URLS = {
    "model": f"{BASE}/w05_model_metrics.json",
    "audit": f"{BASE}/w06_validation_audit.json",
    "playbook": f"{BASE}/w07_action_playbook.json",
}

def load_json(url):
    with urlopen(url) as response:
        return json.load(response)

model_receipt = load_json(URLS["model"])
audit_receipt = load_json(URLS["audit"])
playbook_receipt = load_json(URLS["playbook"])

assert model_receipt["lane"] == playbook_receipt["lane"]
assert playbook_receipt["human_review_required"] is True
print("Loaded committed W05, W06, and W07 receipts.")
print(f"Pages in action queue: {playbook_receipt['total_pages_ranked']:,}")


## 3. Methodology

### Outcome and features
The outcome is an observed future-window proxy: April impressions below 80% of March impressions. It does not measure whether a refresh would cause recovery. All five features end on March 31; April information is used only to construct the outcome.

### Baseline
The hand-written baseline gives two points for content age ≥180 days, two points for March impressions ≥500, and one-point bonuses for age ≥365 days and impressions ≥5,000. It ranks by score, then visibility and age.

### Model and validation
Depth-2 and depth-3 `DecisionTreeClassifier` candidates use `min_samples_leaf=200` and random seed 42. Clients, not rows, are split 60/20/20 into train, validation, and test groups. Depth is selected on validation Precision@50 with average precision as the tie-breaker, then the chosen depth-2 tree is refit on train plus validation clients and evaluated once on held-out test clients.

### Leakage checks
No future fields, labels, IDs, or product decision flags are features, and clients do not cross splits. One population caveat remains: requiring April client coverage uses outcome-window information to define the evaluated population. This does not reveal a page's label to the model, but it narrows the population and is disclosed as a limitation.

## 4. Results

The model beats the hand rule on the same grouped test population, but the base rate changes sharply across client groups and the top-ranked model results do not beat the test base rate. The honest result is therefore limited discrimination, not production-ready prediction.

In [ ]:
baseline = model_receipt["test_baseline"]
model = model_receipt["test_model"]
validation = model_receipt["validation_candidates"][0]

results = pd.DataFrame([
    {
        "Evaluation": "Validation — model",
        "Rows": validation["rows"],
        "Base rate": validation["base_rate"],
        "P@20": validation["precision_at_20"],
        "P@50": validation["precision_at_50"],
        "Average precision": validation["average_precision"],
        "ROC-AUC": validation["roc_auc"],
    },
    {
        "Evaluation": "Grouped test — rule baseline",
        "Rows": baseline["rows"],
        "Base rate": baseline["base_rate"],
        "P@20": baseline["precision_at_20"],
        "P@50": baseline["precision_at_50"],
        "Average precision": baseline["average_precision"],
        "ROC-AUC": baseline["roc_auc"],
    },
    {
        "Evaluation": "Grouped test — depth-2 tree",
        "Rows": model["rows"],
        "Base rate": model["base_rate"],
        "P@20": model["precision_at_20"],
        "P@50": model["precision_at_50"],
        "Average precision": model["average_precision"],
        "ROC-AUC": model["roc_auc"],
    },
])
display(results.round(3))

labels = ["Precision@20", "Precision@50", "Average precision", "ROC-AUC"]
baseline_values = [baseline["precision_at_20"], baseline["precision_at_50"], baseline["average_precision"], baseline["roc_auc"]]
model_values = [model["precision_at_20"], model["precision_at_50"], model["average_precision"], model["roc_auc"]]
x = np.arange(len(labels))
width = 0.34
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, baseline_values, width, label="Rule baseline", color="#94a3b8")
ax.bar(x + width/2, model_values, width, label="Depth-2 tree", color="#2563eb")
ax.axhline(model["base_rate"], color="#ef4444", linestyle="--", label="Test base rate")
ax.set_ylim(0, 0.70)
ax.set_ylabel("Score")
ax.set_title("Grouped test: model improves on the rule, but top-K remains below base rate")
ax.set_xticks(x, labels)
ax.legend()
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
importance = pd.DataFrame(model_receipt["feature_importance"]).sort_values("importance")
display(importance.sort_values("importance", ascending=False).round(3))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(importance["feature"], importance["importance"], color="#0f766e")
ax.set_xlim(0, 0.60)
ax.set_xlabel("Decision-tree importance")
ax.set_title("The fitted tree relies almost entirely on coverage and CTR")
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

print("Takeaway: observed_days_march and ctr_march account for essentially all fitted-tree importance.")


## 5. Limitations & honest framing

1. **The model does not beat the test base rate at the top of the queue.** Precision@50 is 0.38 versus a 0.558 positive base rate.
2. **Cross-client behavior is unstable.** Validation base rate is 0.339 and test base rate is 0.558, a 0.218 shift; validation Precision@50 is only 0.06.
3. **Coverage may be acting as a portfolio proxy.** March observed days and CTR account for essentially all fitted-tree importance; impressions, position, and age are unused by the shallow tree.
4. **Population selection uses outcome-window availability.** Requiring sufficient April client coverage narrows the evaluated population using future information.
5. **The label is a proxy.** Missing April page rows are treated as zero impressions when the client has sufficient April coverage, which may mix genuine decline, removal, and sparse page observation.
6. **The action rule is not causal.** Age and visibility identify review candidates; they do not prove that refreshing will improve search performance.
7. **The study covers one March-to-April transition.** Seasonality and other periods may behave differently.

These limits change the operating decision: use the results for manual triage and further measurement, not automated publishing or pruning.

## 6. Ranked recommendations

Because the learned model is not stable enough for automation, the operational queue uses the transparent March-only rule and mandatory human review.

In [ ]:
action_order = ["REVIEW_NOW", "REVIEW_NEXT", "MONITOR", "DEFER"]
action_descriptions = {
    "REVIEW_NOW": "Score 5–6: very stale or high-volume pages that satisfy both core conditions",
    "REVIEW_NEXT": "Score 4 and average position ≤20: stale, visible, and near meaningful visibility",
    "MONITOR": "Score 4 and average position >20: rule fires, but immediate opportunity is less clear",
    "DEFER": "Score below 4: does not satisfy both age and visibility conditions",
}
counts = playbook_receipt["action_counts"]
total = playbook_receipt["total_pages_ranked"]
actions = pd.DataFrame([
    {
        "Rank": i + 1,
        "Action": action,
        "Pages": counts[action],
        "Share": counts[action] / total,
        "Use": action_descriptions[action],
    }
    for i, action in enumerate(action_order)
])
display(actions.round({"Share": 4}))

colors = ["#dc2626", "#f59e0b", "#0ea5e9", "#cbd5e1"]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(actions["Action"][::-1], actions["Pages"][::-1], color=colors[::-1])
ax.set_xlabel("Pages")
ax.set_title("Final action queue: most pages are deferred; 16.2% enter active review")
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

print("Human check before action: factual freshness, seasonality, intent, cannibalization, business importance, and the correct editorial action.")
print("Never automate: publishing, deletion, pruning, merging, redirects, or causal claims.")


## 7. Reproducibility

- Repository: [github.com/Cooper30/flyrank-ml-internship](https://github.com/Cooper30/flyrank-ml-internship)
- Data contract: [`w03_data_contract.ipynb`](https://github.com/Cooper30/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)
- Baseline: [`w04_baseline_score.ipynb`](https://github.com/Cooper30/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)
- Model: [`w05_model.ipynb`](https://github.com/Cooper30/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)
- Validation audit: [`w06_validation_audit.ipynb`](https://github.com/Cooper30/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)
- Action playbook: [`w07_action_playbook.ipynb`](https://github.com/Cooper30/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb)
- Random seed: `42`
- Environment: Python, DuckDB, pandas, NumPy, scikit-learn, Matplotlib, Hugging Face Hub

Run the notebooks in order from a fresh clone. Add the gated warehouse READ token to Colab Secrets as `HF_TOKEN`; never place credentials in a notebook cell. Generated page-level CSV queues remain gitignored, while committed JSON receipts preserve the reported metrics and audit decisions.

## 8. Acknowledgments & data credit

Built on the [FlyRank ML Internship dataset](https://flyrank.ai). Thanks to the FlyRank internship team for providing the public-safe warehouse release, starter repository, and methodological guidance.

## ML-12 — Five-minute demo outline

1. **0:00–0:40 — Decision:** An editor needs a trustworthy starting queue, not another dashboard.
2. **0:40–1:30 — Data and target:** March features, April decline proxy, 101,280 model rows, 40 clients.
3. **1:30–2:30 — Baseline and model:** Transparent rule versus depth-2 tree under client-grouped validation.
4. **2:30–3:30 — Honest result:** P@50 improves 0.26 → 0.38, but remains below 0.558 base rate; validation instability matters.
5. **3:30–4:30 — Action playbook:** 13,154 review now, 15,435 review next, 2,875 monitor, 145,274 defer.
6. **4:30–5:00 — Takeaway:** The strongest deliverable is not an overclaimed model; it is a reproducible queue with explicit reasons, human checks, and stop conditions.

## ML-12 — Social post cut

I built a content-refresh prioritization study on the FlyRank ML Internship warehouse. A shallow Decision Tree improved Precision@50 from 0.26 to 0.38 over a transparent rule on held-out client groups—but it still fell below the 0.558 test base rate and was unstable across clients. Instead of hiding that result, I turned it into an honest action playbook: 176,738 pages ranked with reason codes, mandatory human review, and explicit monitoring triggers. Reproducible work matters more than a flattering metric.

## ML-12 — Employer-facing summary

I built an interpretable content-refresh ranking workflow on real, public-safe search data using DuckDB, pandas, and scikit-learn. I designed a future-window label, client-grouped validation, leakage checks, and an evaluation that compares the model with both a rule baseline and the task base rate. When the learned model showed unstable cross-client performance, I converted the result into a transparent human-review playbook instead of overstating its readiness.

In [ ]:
summary = {
    "title": "Can Search Signals Prioritize Content Refresh Reviews?",
    "author": "Batuhan Şahin",
    "lane": model_receipt["lane"],
    "modeling_rows": 101280,
    "clients": 40,
    "test_rows": model["rows"],
    "test_base_rate": model["base_rate"],
    "baseline_precision_at_50": baseline["precision_at_50"],
    "model_precision_at_50": model["precision_at_50"],
    "baseline_roc_auc": baseline["roc_auc"],
    "model_roc_auc": model["roc_auc"],
    "validation_precision_at_50": validation["precision_at_50"],
    "validation_base_rate": validation["base_rate"],
    "total_pages_ranked": total,
    "action_counts": counts,
    "final_operating_mode": playbook_receipt["operating_mode"],
    "automated_actions_allowed": False,
    "random_seed": 42,
}

OUTPUT = Path("work/outputs/capstone_summary.json")
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

assert summary["model_precision_at_50"] > summary["baseline_precision_at_50"]
assert summary["model_precision_at_50"] < summary["test_base_rate"]
assert sum(summary["action_counts"].values()) == summary["total_pages_ranked"]
assert summary["automated_actions_allowed"] is False
print("CAPSTONE CHECK PASSED ✅")
print(f"Summary written to: {OUTPUT}")

from google.colab import files
files.download(str(OUTPUT))


## Final self-check

- [ ] Abstract contains exactly five complete sentences
- [ ] All required paper sections are present
- [ ] Model and baseline are compared on the same grouped test rows
- [ ] Base rates are visible next to ranking metrics
- [ ] Limitations include instability, population selection, label ambiguity, and non-causality
- [ ] Ranked recommendations include reasoned actions and mandatory human review
- [ ] Notebook/repo links and random seed are present
- [ ] FlyRank data credit is linked
- [ ] ML-12 demo, social post, and employer summary are included
- [ ] `CAPSTONE CHECK PASSED ✅` appears